# Analyze Results

This notebook summarizes both simulation and financial real-data results for OT estimators.
It supports three tasks: simulation boxplots, simulation LaTeX table generation, and real-data LaTeX table generation.


In [3]:
import os, re, csv, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
from scipy.stats import norm, t as t_dist

from network import ICNN, compute_gradients

In [ ]:
def load_model(input_size, hidden_size, act, model_path, device):
    model = ICNN(input_size, hidden_size, 1, activation=act).to(device)
    state = torch.load(model_path, map_location=device)
    if isinstance(state, dict) and all(isinstance(v, torch.Tensor) for v in state.values()):
        model.load_state_dict(state)
    elif isinstance(state, dict) and "state_dict" in state:
        model.load_state_dict(state["state_dict"])
    else:
        model.load_state_dict(state)
    model.eval()
    return model


def sign_square(x: torch.Tensor) -> torch.Tensor:
    return torch.sign(x) * (x ** 2)


def l2_evaluate(model, x_P_l2: torch.Tensor, label_P_l2: np.ndarray, device) -> float:
    grad = compute_gradients(model, x_P_l2.to(device)).cpu().numpy()
    grad = np.asarray(grad, dtype=float)
    label = np.asarray(label_P_l2, dtype=float)
    return float(np.mean((label - grad) ** 2) ** 0.5)


def L2_loss(model, transform_method: str, measure_P: str, input_size: int, device) -> float:
    df = 6
    t_distribution = torch.distributions.StudentT(df)

    if measure_P == "normal":
        x_l2 = torch.randn(10_000, input_size, device=device, dtype=torch.float32).requires_grad_(True)
    elif measure_P == "t":
        x_l2 = t_distribution.sample((10_000, input_size)).to(device).requires_grad_(True)
    else:
        raise ValueError(f"Unknown measure_P: {measure_P}")

    if transform_method == "CDF":
        if measure_P == "normal":
            label_l2 = norm.cdf(x_l2.detach().cpu().numpy())
        else:
            label_l2 = t_dist.cdf(x_l2.detach().cpu().numpy(), df)
    elif transform_method == "piecewise_linear":
        z = x_l2.detach()
        abs_z = z.abs()
        sgn_z = torch.sign(z)

        y = torch.empty_like(z)
        mask1 = abs_z <= 1.0
        y[mask1] = z[mask1]

        mask2 = (abs_z > 1.0) & (abs_z <= 2.0)
        y[mask2] = sgn_z[mask2] * (0.5 * (abs_z[mask2] - 1.0) + 1.0)

        mask3 = abs_z > 2.0
        y[mask3] = sgn_z[mask3] * (2.0 * (abs_z[mask3] - 2.0) + 1.5)

        label_l2 = y.cpu().numpy()
    elif transform_method == "quadratic":
        label_l2 = sign_square(x_l2.detach()).cpu().numpy()
    else:
        raise ValueError(f"Unknown transform_method: {transform_method}")

    return l2_evaluate(model, x_l2, label_l2, device=device)


In [ ]:
# Parsers for simulation result folder names
DIR_RE = re.compile(r"^d=(\d+)$")
HP_RE  = re.compile(
    r"^(?P<measure>normal|t)_(?P<transform>CDF|piecewise_linear|quadratic)_n_(?P<n>\d+)_k_(?P<k>-?\d+(?:\.\d+)?)(?:_M1_(?P<m1>-?\d+(?:\.\d+)?))?$"
)


def parse_m1_value(m1_token):
    # No suffix means the legacy/default case M1 = infinity.
    if m1_token is None:
        return float("inf")
    return float(m1_token)


def m1_matches_target(m1_value, target_m1):
    if target_m1 is None:
        return True
    if np.isinf(target_m1):
        return np.isinf(m1_value)
    return np.isfinite(m1_value) and abs(float(m1_value) - float(target_m1)) <= 1e-12


def evaluate_subfolder(
    subfolder_path: str,
    input_size: int,
    hp: dict,
    hidden_size: int = 16,
    act: str = "softplus_scaled",
    device: str = "cpu",
):
    """Evaluate all model_*.pth in one scenario folder and write one CSV into the same folder."""
    measure = hp["measure"]
    transform = hp["transform"]
    m1_value = hp["m1"]

    model_paths = []
    for idx in range(100):
        mp = os.path.join(subfolder_path, f"model_{idx}.pth")
        if os.path.exists(mp):
            model_paths.append((idx, mp))

    if not model_paths:
        print(f"[skip] no models found in: {subfolder_path}")
        return

    rows = []
    use_device = torch.device(device if (device == "cpu" or torch.cuda.is_available()) else "cpu")

    for idx, mp in model_paths:
        try:
            model = load_model(input_size, hidden_size, act, mp, use_device)
            loss = float(L2_loss(model, transform, measure, input_size, use_device))
            rows.append((idx, loss))
        except Exception as e:
            print(f"[warn] failed on {mp}: {e}")

    out_csv = os.path.join(
        subfolder_path,
        f"L2_error_measure_P={measure}_transform_method={transform}.csv",
    )

    with open(out_csv, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["model_idx", "L2_loss", "input_size", "measure_P", "transform_method", "sample_size", "k", "M1"])
        for idx, loss in sorted(rows):
            m1_text = "inf" if np.isinf(m1_value) else f"{m1_value:.12g}"
            w.writerow([idx, f"{loss:.12g}", input_size, measure, transform, hp["n"], hp["k"], m1_text])

    print(f"[done] {out_csv}  (rows: {len(rows)})")


def evaluate_all(
    root_dir: str,
    hidden_size: int = 16,
    act: str = "softplus_scaled",
    device: str = "cpu",
    target_m1: float = float("inf"),
):
    """
    Walk root_dir/d=<d>/<scenario>/ and evaluate every recognized scenario folder.

    target_m1:
      - inf   -> only legacy/default folders (no _M1 suffix)
      - value -> only folders with _M1_<value>
      - None  -> evaluate all folders regardless of M1
    """
    for dname in sorted(os.listdir(root_dir)):
        m = DIR_RE.match(dname)
        if not m:
            continue

        input_size = int(m.group(1))
        dpath = os.path.join(root_dir, dname)
        if not os.path.isdir(dpath):
            continue

        for sub in sorted(os.listdir(dpath)):
            spath = os.path.join(dpath, sub)
            if not os.path.isdir(spath):
                continue

            m2 = HP_RE.match(sub)
            if not m2:
                continue

            m1_value = parse_m1_value(m2.group("m1"))
            if not m1_matches_target(m1_value, target_m1):
                continue

            hp = m2.groupdict()
            hp["n"] = int(hp["n"])
            hp["k"] = float(hp["k"])
            hp["m1"] = m1_value

            m1_label = "inf" if np.isinf(m1_value) else f"{m1_value:.12g}"
            print(f"[eval] d={input_size}  {sub} (M1={m1_label})")
            evaluate_subfolder(spath, input_size, hp, hidden_size=hidden_size, act=act, device=device)


In [ ]:
# Configure paths / model settings, then run evaluation
ROOT = "../simulation_results"
TARGET_M1 = float("inf")  # use e.g. 10.0 for finite M1 folders, or None for all

# evaluate_all(ROOT, hidden_size=16, act="softplus_scaled", device="cpu", target_m1=TARGET_M1)


## Simulation Boxplots

This section reads simulation CSV outputs and generates side-by-side boxplots comparing Dual-type and Sieve estimators across settings.


In [ ]:
FLOAT_RE = re.compile(r"[-+]?\d*\.\d+|\d+")


def read_losses_from_csv(csv_path, loss_column="L2_loss"):
    df = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
    if loss_column and loss_column in df.columns:
        return df[loss_column].astype(float).to_numpy()
    try:
        return df.iloc[:, 0].astype(float).to_numpy()
    except Exception:
        text = open(csv_path, "r").read()
        nums = FLOAT_RE.findall(text)
        return np.array([float(x) for x in nums], dtype=float)


def plot_k_comparison_per_d(
    root_dir,
    d,
    csv_pattern="*.csv",
    loss_column="L2_loss",
    pair_gap=2,
    box_width=0.7,
    legend_fontsize=11,
    save=False,
    outdir=None,
    target_m1=float("inf"),
):
    pat = re.compile(
        r"^(normal|t)_(CDF|piecewise_linear|quadratic)_(?:sample_size_|n_)(100|300|500|1000)_k_(-?\d+(?:\.\d+)?)(?:_M1_(-?\d+(?:\.\d+)?))?$"
    )

    order_dist = ["normal", "t"]
    order_method = ["CDF", "piecewise_linear", "quadratic"]
    sample_sizes = [100, 300, 500, 1000]
    order_k = [1.0, 2.0, -1.0]

    colors = {1.0: "orange", 2.0: "red", -1.0: "blue"}
    labels = {1.0: "Sieve, k=1", 2.0: "Sieve, k=2", -1.0: "Dual-type"}

    dist_display = {"normal": "Normal", "t": "t(6)"}
    method_display = {"CDF": "rank", "piecewise_linear": "piecewise_linear", "quadratic": "quadratic"}

    d_dir = os.path.join(root_dir, f"d={d}")
    if not os.path.isdir(d_dir):
        print(f"[skip] {d_dir} not found")
        return

    data = {}
    for name in os.listdir(d_dir):
        sub = os.path.join(d_dir, name)
        if not os.path.isdir(sub):
            continue

        m = pat.match(name)
        if not m:
            continue

        dist, method, N_str, k_str, m1_str = m.groups()
        m1_value = parse_m1_value(m1_str)
        if not m1_matches_target(m1_value, target_m1):
            continue

        N = int(N_str)
        kv = float(k_str)

        files = glob.glob(os.path.join(sub, csv_pattern))
        if not files:
            continue

        losses = read_losses_from_csv(files[0], loss_column)
        data.setdefault(dist, {}).setdefault(method, {}).setdefault(N, {})[kv] = losses

    fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey=False)
    plt.subplots_adjust(wspace=0.3, hspace=0.4)

    group_span = len(order_k) + pair_gap
    offsets = {1.0: 1, 2.0: 2, -1.0: 3}

    for i, dist in enumerate(order_dist):
        for j, method in enumerate(order_method):
            ax = axes[i][j]

            all_positions = {k: [] for k in order_k}
            all_boxes = {k: [] for k in order_k}

            for g, N in enumerate(sample_sizes):
                base = g * group_span
                for kv in order_k:
                    all_positions[kv].append(base + offsets[kv])
                    arr = data.get(dist, {}).get(method, {}).get(N, {}).get(kv, np.array([]))
                    all_boxes[kv].append(arr)

            for kv in order_k:
                ax.boxplot(
                    all_boxes[kv],
                    positions=all_positions[kv],
                    widths=box_width,
                    patch_artist=True,
                    boxprops=dict(facecolor=colors[kv], alpha=0.7),
                    medianprops=dict(color="black"),
                    whiskerprops=dict(color="black"),
                    capprops=dict(color="black"),
                    flierprops=dict(marker="o", markersize=3, markerfacecolor=colors[kv], alpha=0.6),
                )

            centers = [g * group_span + 2 for g in range(len(sample_sizes))]
            ax.set_xticks(centers)
            ax.set_xticklabels(sample_sizes)
            ax.set_xlabel("sample size")
            ax.set_title(f"{dist_display[dist]} - {method_display[method]}")

            ax.set_yscale("log")
            vals = []
            for kv in order_k:
                for arr in all_boxes[kv]:
                    if arr.size:
                        vals.append(arr)
            if vals:
                vals = np.concatenate(vals)
                ymin, ymax = vals.min(), vals.max()
                ax.set_ylim(max(ymin * 0.8, 1e-12), min(ymax * 1.25, 20))

            for g in range(len(sample_sizes) - 1):
                x_sep = g * group_span + (offsets[-1.0] + group_span + offsets[1.0]) / 2
                ax.axvline(x=x_sep, color="gray", linestyle="--", linewidth=1, alpha=0.7, zorder=0)

            if i == 0 and j == 0:
                handles = [mpatches.Patch(color=colors[kv], label=labels[kv], alpha=0.7) for kv in order_k]
                ax.legend(handles=handles, loc="upper right", fontsize=legend_fontsize, frameon=False)

            ax.grid(axis="y", linestyle="--", alpha=0.5)
            ax.set_ylabel("loss (log scale)")

    plt.tight_layout(rect=[0, 0, 1, 0.95])

    if save:
        os.makedirs(outdir or root_dir, exist_ok=True)
        out_path = os.path.join(outdir or root_dir, f"loss_grid_d{d}.pdf")
        plt.savefig(out_path, dpi=200)
        print(f"Saved: {out_path}")
    else:
        plt.show()


In [ ]:
# Example plotting call (adjust paths if needed)
# for d_val in (10, 20):
#     plot_k_comparison_per_d(ROOT, d=d_val, loss_column="L2_loss", save=True, outdir="../OT_results_analysis", target_m1=TARGET_M1)


## Simulation LaTeX Tables

This section aggregates simulation results and prints LaTeX tables for direct insertion into the manuscript.


In [ ]:
from typing import Dict, Tuple, List, Sequence, Union

Key = Tuple[str, str, int, int]  # (measure, transform, n, d)


def _load_numeric_series_from_csv(path: str) -> np.ndarray:
    vals: List[float] = []
    with open(path, "r", newline="") as f:
        reader = csv.reader(f)
        header = next(reader, None)
        col_idx = None
        if header:
            lower = [h.lower() if isinstance(h, str) else "" for h in header]
            for name in ("test_rmse", "rmse", "l2_loss", "loss", "value"):
                if name in lower:
                    col_idx = lower.index(name)
                    break
            if col_idx is None:
                col_idx = 1 if len(lower) > 1 else 0

        for row in reader:
            if not row:
                continue
            cell = row[col_idx] if (col_idx is not None and col_idx < len(row)) else row[0]
            try:
                v = float(cell)
                if np.isfinite(v):
                    vals.append(v)
            except Exception:
                pass

    return np.asarray(vals, dtype=float)


def load_single_estimator_for_k(orig_root: str, k_text: str, target_m1: float = float("inf")) -> Dict[Key, np.ndarray]:
    data: Dict[Key, List[np.ndarray]] = {}
    for dname in os.listdir(orig_root):
        dm = DIR_RE.match(dname)
        if not dm:
            continue

        d = int(dm.group(1))
        dpath = os.path.join(orig_root, dname)
        if not os.path.isdir(dpath):
            continue

        for sub in os.listdir(dpath):
            spath = os.path.join(dpath, sub)
            if not os.path.isdir(spath):
                continue

            m = HP_RE.match(sub)
            if not m or m.group("k") != k_text:
                continue

            m1_value = parse_m1_value(m.group("m1"))
            if not m1_matches_target(m1_value, target_m1):
                continue

            measure = m.group("measure")
            transform = m.group("transform")
            n = int(m.group("n"))

            series_list: List[np.ndarray] = []
            for fn in os.listdir(spath):
                if fn.lower().endswith(".csv"):
                    arr = _load_numeric_series_from_csv(os.path.join(spath, fn))
                    if arr.size > 0:
                        series_list.append(arr)

            if series_list:
                key: Key = (measure, transform, n, d)
                data.setdefault(key, []).extend(series_list)

    out: Dict[Key, np.ndarray] = {}
    for key, lst in data.items():
        out[key] = np.concatenate(lst, axis=0)
    return out


def _normalize_k_text(k: Union[str, float, int]) -> str:
    if isinstance(k, str):
        return k
    return f"{float(k):.1f}"


def _stats(arr: np.ndarray, upper_clip=None):
    if arr is None or arr.size == 0:
        return np.nan, np.nan

    vals = np.asarray(arr, dtype=float)
    mask = np.isfinite(vals)
    if upper_clip is not None:
        mask &= vals <= upper_clip
    vals = vals[mask]

    if vals.size == 0:
        return np.nan, np.nan

    mean = float(vals.mean())
    sd = float(vals.std(ddof=1)) if vals.size > 1 else np.nan
    return mean, sd


def summarize_dimension_table_for_ks(
    orig_root: str,
    d_target: int,
    k_values: Sequence[Union[str, float, int]] = ("-1.0", "1.0", "2.0"),
    Ns: Sequence[int] = (100, 300, 500, 1000),
    upper_clip=None,
    target_m1: float = float("inf"),
) -> np.ndarray:
    """
    Build an 18x8 table for one dimension and three k settings:
      rows per k:
        (normal,CDF), (normal,piecewise_linear), (normal,quadratic),
        (t,CDF),      (t,piecewise_linear),      (t,quadratic)
      columns:
        [mean@100, sd@100, mean@300, sd@300, mean@500, sd@500, mean@1000, sd@1000]
    """
    cases = [
        ("normal", "CDF"),
        ("normal", "piecewise_linear"),
        ("normal", "quadratic"),
        ("t", "CDF"),
        ("t", "piecewise_linear"),
        ("t", "quadratic"),
    ]

    Ns = [int(n) for n in Ns]
    rows: List[List[float]] = []

    for k in k_values:
        k_text = _normalize_k_text(k)
        k_map = load_single_estimator_for_k(orig_root, k_text, target_m1=target_m1)

        for measure, transform in cases:
            row: List[float] = []
            for n in Ns:
                arr = k_map.get((measure, transform, int(n), int(d_target)), None)
                mu, sd = _stats(arr, upper_clip=upper_clip)
                row.extend([mu, sd])
            rows.append(row)

    return np.array(rows, dtype=float)


def table18x8_to_latex_ot_p_estimator(
    A: np.ndarray,
    estimator_labels: Sequence[str],
    digits: int = 4,
    sample_sizes: Sequence[int] = (100, 300, 500, 1000),
    t_df_label: int = 6,
    nan_token: str = "--",
) -> str:
    A = np.asarray(A, dtype=float)
    L = len(estimator_labels)
    if A.shape != (6 * L, 8):
        raise ValueError(f"Expected A with shape ({6 * L}, 8), got {A.shape}")

    def fmt(x, bold=False):
        if x is None or not np.isfinite(x) or np.isnan(x):
            return nan_token
        s = f"{float(x):.{digits}f}"
        return rf"\\textbf{{{s}}}" if bold else s

    def row_idx(est_idx, case_idx):
        return 6 * est_idx + case_idx

    P_names = {
        "normal": r"$N(0,1)$",
        "t": rf"$t({t_df_label})$",
    }
    OT_names = {
        "CDF": "Rank function",
        "piecewise_linear": "Linear",
        "quadratic": "Sign-quadratic",
    }

    cases = [
        ("normal", "CDF"),
        ("normal", "piecewise_linear"),
        ("normal", "quadratic"),
        ("t", "CDF"),
        ("t", "piecewise_linear"),
        ("t", "quadratic"),
    ]
    case_to_idx = {c: i for i, c in enumerate(cases)}

    ot_order = ["CDF", "piecewise_linear", "quadratic"]
    p_order = ["normal", "t"]
    ss = list(sample_sizes)

    winner = {}
    for measure in p_order:
        for transform in ot_order:
            cidx = case_to_idx[(measure, transform)]
            for j in range(len(ss)):
                mean_col = 2 * j
                means = np.array([A[row_idx(e, cidx), mean_col] for e in range(L)], dtype=float)
                valid = np.isfinite(means)
                if not valid.any():
                    winner[(measure, transform, j)] = None
                else:
                    idx_valid = np.where(valid)[0]
                    best_local = idx_valid[np.argmin(means[valid])]
                    winner[(measure, transform, j)] = int(best_local)

    lines = [r"\\begin{tabular}{ccc|cc|cc|cc|cc}", r"\\hline"]

    group_cells = []
    for i, n in enumerate(ss):
        bar = "|" if i < len(ss) - 1 else ""
        group_cells.append(rf"\\multicolumn{{2}}{{c{bar}}}{{$n=N={int(n)}$}}")
    lines.append(r"& & & " + " & ".join(group_cells) + r" \\\\ \\hline")

    lines.append(
        r"OT map & $P$ & Estimator & "
        + " & ".join(["Mean", "SD"] * len(ss))
        + r" \\\\ \\hline"
    )

    for transform in ot_order:
        for measure in p_order:
            cidx = case_to_idx[(measure, transform)]

            for est_pos, est_label in enumerate(estimator_labels):
                vals = []
                for j in range(len(ss)):
                    r = row_idx(est_pos, cidx)
                    mu = A[r, 2 * j]
                    sd = A[r, 2 * j + 1]
                    is_best = winner[(measure, transform, j)] == est_pos
                    vals.append(fmt(mu, bold=is_best))
                    vals.append(fmt(sd, bold=is_best))

                nums = " & ".join(vals)

                if measure == "normal" and est_pos == 0:
                    lines.append(
                        rf"\\multirow{{6}}{{*}}{{{OT_names[transform]}}} "
                        rf"& \\multirow{{3}}{{*}}{{{P_names[measure]}}} "
                        rf"& {est_label} & {nums} \\\\"
                    )
                elif measure == "t" and est_pos == 0:
                    lines.append(r"\\cline{2-11}")
                    lines.append(
                        rf" & \\multirow{{3}}{{*}}{{{P_names[measure]}}} "
                        rf"& {est_label} & {nums} \\\\"
                    )
                else:
                    lines.append(f" & & {est_label} & {nums} \\")

            if measure == "t":
                lines.append(r"\\hline")

    lines.append(r"\\end{tabular}")
    return "\n".join(lines)



In [ ]:
# Example LaTeX table generation
TABLE_ROOT = ROOT

estimator_labels = ["Dual-type", r"Sieve $k=2$", r"Sieve $k=1$"]

# for d in (10, 20):
#     A = summarize_dimension_table_for_ks(
#         orig_root=TABLE_ROOT,
#         d_target=d,
#         k_values=("-1.0", "2.0", "1.0"),
#         Ns=(100, 300, 500, 1000),
#         upper_clip=None,
#         target_m1=TARGET_M1,
#     )
#
#     latex = table18x8_to_latex_ot_p_estimator(
#         A=A,
#         estimator_labels=estimator_labels,
#         digits=4,
#         sample_sizes=(100, 300, 500, 1000),
#         t_df_label=6,
#     )
#
#     print(f"\n===== d={d} =====")
#     print(latex)


## Real-Data LaTeX Table

This section reads the finance evaluation CSV files (`mmd_average_over_models.csv` and `sw2_average_over_models.csv`) and prints the LaTeX `tabular` for the real-data comparison table.
Each metric is reported as `mean (bootstrap sd)`.


In [6]:
# Real-data LaTeX table generation (finance)
# Required files:
#   model_finance/evaluation/mmd_average_over_models.csv
#   model_finance/evaluation/sw2_average_over_models.csv

from pathlib import Path

mmd_path = Path("model_finance/evaluation/mmd_average_over_models.csv")
sw2_path = Path("model_finance/evaluation/sw2_average_over_models.csv")

mmd = pd.read_csv(mmd_path)
sw2 = pd.read_csv(sw2_path)

df = mmd.merge(sw2, on=["pair", "k", "M1", "n_evals"], how="inner")

# Keep manuscript rows:
# Dual-type: k=-1, M1=inf
# Sieve: k=1 or 2, M1=32
m1_num = pd.to_numeric(df["M1"], errors="coerce")
mask_dual = (df["k"] == -1.0) & (df["M1"].astype(str).str.lower() == "inf")
mask_sieve = (df["k"].isin([1.0, 2.0])) & (m1_num == 32.0)
df = df[mask_dual | mask_sieve].copy()

# Ensure SD columns exist
if "mmd_rbf_bootstrap_sd" not in df.columns:
    df["mmd_rbf_bootstrap_sd"] = np.nan
if "sw2_bootstrap_sd" not in df.columns:
    df["sw2_bootstrap_sd"] = np.nan

pair_display = {
    "2021bull_to_2022bear": r"2021 bull $\to$ 2022 bear",
    "2024bull_to_2025bear": r"2024 bull $\to$ 2025 bear",
}
est_display = {
    -1.0: "Dual-type",
    1.0: r"Sieve $k=1$",
    2.0: r"Sieve $k=2$",
}

period_order = ["2021bull_to_2022bear", "2024bull_to_2025bear"]
est_order = [-1.0, 1.0, 2.0]


def fmt_mean_sd(mean_val, sd_val, digits=5):
    m = f"{float(mean_val):.{digits}f}" if pd.notna(mean_val) else "nan"
    s = f"{float(sd_val):.{digits}f}" if pd.notna(sd_val) else "nan"
    return f"{m} ({s})"


# Build a lookup dict for easy formatting
key_cols = ["pair", "k"]
lookup = {
    (r["pair"], float(r["k"])): r
    for _, r in df.iterrows()
}

lines = []
lines.append(r"\begin{tabular}{cc|cc}")
lines.append(r"\hline")
lines.append(r"    Period                       & Estimator   & MMD & SWD \\ \hline")

for pair in period_order:
    rows = []
    for k in est_order:
        r = lookup[(pair, k)]
        rows.append({
            "k": k,
            "Estimator": est_display[k],
            "MMD": fmt_mean_sd(r["mmd_rbf"], r["mmd_rbf_bootstrap_sd"]),
            "SWD": fmt_mean_sd(r["sw2"], r["sw2_bootstrap_sd"]),
            "mmd_val": float(r["mmd_rbf"]),
            "swd_val": float(r["sw2"]),
        })

    # Winner by mean metric (lower is better)
    best_mmd_k = min(rows, key=lambda x: x["mmd_val"])["k"]
    best_swd_k = min(rows, key=lambda x: x["swd_val"])["k"]

    for i, row in enumerate(rows):
        est = row["Estimator"]
        mmd_txt = row["MMD"]
        swd_txt = row["SWD"]

        # Bold estimator label only when both metrics pick the same winner,
        # matching the style in your example table.
        if (best_mmd_k == best_swd_k) and (row["k"] == best_mmd_k):
            est = r"\textbf{" + est + "}"

        if row["k"] == best_mmd_k:
            mmd_txt = r"\textbf{" + mmd_txt + "}"
        if row["k"] == best_swd_k:
            swd_txt = r"\textbf{" + swd_txt + "}"

        if i == 0:
            lines.append(
                r"\multirow{3}{*}{" + pair_display[pair] + r"} & "
                + est + " & " + mmd_txt + " & " + swd_txt + r" \\"
            )
        else:
            lines.append(
                "                                 & "
                + est + " & " + mmd_txt + " & " + swd_txt + r" \\"
            )

    lines.append(r"\hline")

lines.append(r"\end{tabular}")

latex_table = "\n".join(lines)
print("===== Real-Data LaTeX Table (mean with bootstrap sd) =====\n")
print(latex_table)


===== Real-Data LaTeX Table (mean with bootstrap sd) =====

\begin{tabular}{cc|cc}
\hline
    Period                       & Estimator   & MMD & SWD \\ \hline
\multirow{3}{*}{2021 bull $\to$ 2022 bear} & Dual-type & 0.019025 (0.006375) & 0.815463 (0.109181) \\
                                 & Sieve $k=1$ & 0.006713 (0.003099) & 0.420748 (0.044151) \\
                                 & \textbf{Sieve $k=2$} & \textbf{0.006337 (0.003347)} & \textbf{0.414655 (0.047679)} \\
\hline
\multirow{3}{*}{2024 bull $\to$ 2025 bear} & Dual-type & 0.005652 (0.007804) & 0.552859 (0.154534) \\
                                 & \textbf{Sieve $k=1$} & \textbf{0.000563 (0.006450)} & \textbf{0.469700 (0.165726)} \\
                                 & Sieve $k=2$ & 0.002153 (0.007726) & 0.490580 (0.169112) \\
\hline
\end{tabular}
